# Stable diffusion process
## Imports and setups

In [1]:
import os
from base64 import b64encode

import numpy as np
import torch
from torch import autocast
from torchvision import transforms as trms
from diffusers import AutoencoderKL, LMSDiscreteScheduler, UNet2DConditionModel
from huggingface_hub import notebook_login

# for video display
from IPython.display import HTML
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

from transformers import CLIPTextModel, CLIPTokenizer, logging

from torch import autocast
from torchvision import transforms as tfms

/opt/homebrew/Caskroom/miniforge/base/envs/transformers/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
torch.manual_seed(0)
if not (Path.home()/'.cache/hugginggface'/'token').exists(): notebook_login()

# supress some unnecesary warnings when loading the CLIPTextModel
logging.set_verbosity_error()

# set device
torch_device = "cuda" if torch.device.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
if "mps" == torch_device: os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# Loading models

In [ ]:
# load autoencoder model to use to decode the latents into image space
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae")
# load tokenizer and text encoder to tokenize and encode text
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")
# the UNet model for generating the latents
unet = UNet2DConditionModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="unet")
# Noise Scheduler
scheduler = LMSDiscreteScheduler(
    beta_start=0.00085,
    beta_end=0.012,
    beta_schedule="scaled_linear",
    num_train_timesteps=1000
)

# load above to GPU
vae = vae.to(torch_device)
text_encoder = text_encoder.to(torch_device)
unet = unet.to(torch_device)

## Diffusion Loop 

In [ ]:
def set_timesteps(scheduler, num_inference_steps):
    scheduler.set_timesteps(num_inference_steps)
    scheduler.timesteps = scheduler.timesteps.to(torch.float32)

In [ ]:
# define some initial settings
prompt = ["A watercolor paiting of an otter"]
height = 512                        # default height for stable diffusion
width = 512                         # default width for stable diffusion
num_inference_steps = 30            # num of denoising steps
guidance_scale = 7.5                # scale for classifier-free guidance
generator = torch.manual_seed(32)   # seed for initial latent noise
batch_size = 1

# prepare text
text_input = tokenizer(
    prompt,
    padding="max-length",
    max_length=tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt"
)

with torch.no_grad():
    text_embeddings = text_encoder(text_input.input_ids.to(torch_device))

max_length = text_input.input_ids.shape[-1]
uncond_input = tokenizer(
    [""]*batch_size,
    padding="max_length",
    max_length=max_length,
    return_tensors="pt"
)

with torch.no_grad():
    uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))

text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

# prep scheduler
set_timesteps(scheduler, num_inference_steps)

# prepare scheduler
latents = torch.randn(
    (batch_size, unet.in_channels, height // 8, width // 8),
    generator=generator
)
latents = latents.to(torch_device)
latents = latents * scheduler.init_noise_sigma

# now loop
with autocast("cuda"):
    for i, t in tqdm(enumerate(scheduler.timesteps), total=len(scheduler.timesteps)):
        # when doing classifier-free guidance, need to expand the latents
        latent_model_input = torch.cat([latents]*2)
        sigma = scheduler.sigmas[1]
        # scale latents (preconditioning):
        # latent_model_input = latent_model_input / ((sigma**2 + 1) ** 0.5) # Diffusers 0.3 and below
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        # predict noise residual
        with torch.no_grad():
            noise_pred = unet(
                latent_model_input,
                t,
                encoder_hidden_states=text_embeddings
            ).sample
        
        # perform guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # compute the previous noise sample x_t -> x_t-1
        # latents = scheduler.step(noise_pred, i, latents)["prev_sample"] # diffusers 0.3 and below
        latents = scheduler.step(noise_pred, t, latents).prev_sample

# scale and decode the image latents with vae
latents = 1 / 0.18215 * latents
with torch.no_grad():
    image = vae.decode(latents).sample

# display image
image = (image / 2 + 0.5).clamp(0, 1)
image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
images = (image * 255).round().astype("uint8")
pil_images = [Image.fromarray(image) for image in images]
pil_images[0]

## the AutoEncoder part
Creates a latent representation of the image, and then decodes it back by optimizing the reconstruction error.

In [ ]:
def pil_to_latent(input_image):
    # single image -> single latent in a batch (1, 4, 64, 64)
    with torch.no_grad():
        latent = vae.encode(
            tfms.ToTensor()(input_image).unsqueeze(0).to(torch_device)*2-1 # here it scales
        )
        return 0.18215 * latent.latent_dist.sample()

def latents_to_pil(latents):
    # batch of latents -> list of images
    with torch.no_grad():
        image = vae.decode(latents).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
    imgaes = (image * 255).round().astype("uint8")
    pil_images = [Image.fromarray(image) for image in images]

    return pil_images

Test previous functions and see how it does

In [ ]:
# download an image from the web
!curl --output macaw.jpg 'https://lafeber.com/pet-birds/wp-content/uploads/2018/06/Scarlet-Macaw-2.jpg'

In [ ]:
# load image with PIL
input_image = Image.open("macaw.jpg").resize((512, 512))
input_image

In [ ]:
# encode into latent space
encoded = pil_to_latent(input_image)
encoded.shape

In [ ]:
# visualize the 4 channels of this latent representation
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for c in range(4):
    axs[c].imshow(encoded[0][c].cpu(), cmap="Greys")

Here it can be seen that the latent representation really captures a lot of information, so it can handle the reconstruction almost as equal as original image.

In [ ]:
# create the image back from latents
decoded = latents_to_pil(encoded)[0]
decoded

## The scheduler
This is the part where noise is going to be added. The ammount of noise added follows a certain distribution.

In [ ]:
# set number of sampling steps
set_timesteps(scheduler, 15)

# look at how much noise is added at each step
print(scheduler.sigmas)

In [ ]:
# see how it would look like in an image
noise = torch.rand_like(encoded)
sampling_step = 10
encoded_and_noised = scheduler.add_noise(
    encoded,
    noise,
    timesteps=torch.tensor(
        [scheduler.timesteps[sampling_step]]
    )
)
latents_to_pil(encoded_and_noised.float())[0]

# Image2Image.
Here the idea is to noise an original image, then based on some prompt denoise it and create a new one.

In [ ]:
# Settings (same as before except for the new prompt)
prompt = ["A colorful dancer, nat geo photo"]
height = 512                        # default height of Stable Diffusion
width = 512                         # default width of Stable Diffusion
num_inference_steps = 50            # Number of denoising steps
guidance_scale = 8                  # Scale for classifier-free guidance
generator = torch.manual_seed(32)   # Seed generator to create the inital latent noise
batch_size = 1

# need to preprare text same as done before
